# E8 in 5 Minutes: DHL-MM Sparse Lie Algebra Engine

This notebook demonstrates the DHL-MM engine for all five exceptional Lie algebras:
G2, F4, E6, E7, E8. We show sparse structure constants, Jacobi identity verification,
benchmarking, differentiable PyTorch brackets, and equivariance verification.

In [ ]:
# Install (uncomment for Colab)
# !pip install numpy torch scipy
# !pip install git+https://github.com/grapheneaffiliate/DHL-MM.git

import numpy as np
import sys
sys.path.insert(0, '..')  # if running from notebooks/ directory

## All 5 Exceptional Algebras

DHL-MM precomputes sparse structure constants for each algebra.
The compression ratio shows how much sparser the structure constants are compared to the full dim^3 tensor.

In [ ]:
from exceptional import ExceptionalAlgebra

for name in ['G2', 'F4', 'E6', 'E7', 'E8']:
    alg = ExceptionalAlgebra(name)
    print(f"{name}: dim={alg.dim}, roots={alg.n_roots}, "
          f"sparse_entries={alg.n_structure_constants}, "
          f"compression={alg.dim**3 / alg.n_structure_constants:.1f}x")

## Verify Jacobi Identity

The Jacobi identity [x,[y,z]] + [y,[z,x]] + [z,[x,y]] = 0 must hold exactly
(to machine precision) for any Lie algebra. This is a strong correctness check.

In [ ]:
alg = ExceptionalAlgebra("E8")
rng = np.random.RandomState(42)
x, y, z = rng.randn(248), rng.randn(248), rng.randn(248)

jac = (alg.bracket(x, alg.bracket(y, z)) + 
       alg.bracket(y, alg.bracket(z, x)) + 
       alg.bracket(z, alg.bracket(x, y)))

print(f"Jacobi identity violation: {np.max(np.abs(jac)):.2e}")
print(f"Machine epsilon: {np.finfo(np.float64).eps:.2e}")
print("Exact to machine precision!" if np.max(np.abs(jac)) < 1e-12 else "ERROR")

## Benchmark: Sparse vs Dense Bracket

The sparse bracket uses only the nonzero structure constants, avoiding the full dim^3 tensor.
For E8 (dim=248), this gives a massive speedup.

In [ ]:
import time

alg = ExceptionalAlgebra("E8")
x, y = rng.randn(248), rng.randn(248)

# Sparse bracket
t0 = time.perf_counter()
for _ in range(10000):
    alg.bracket(x, y)
t_sparse = (time.perf_counter() - t0) / 10000

# Dense: build adjoint matrices and multiply
gen = np.array(alg.gen_array)  # (248, 248, 248)
t0 = time.perf_counter()
for _ in range(1000):
    X = np.einsum('i,iab->ab', x, gen)
    Y = np.einsum('i,iab->ab', y, gen)
    XY = X @ Y - Y @ X
t_dense = (time.perf_counter() - t0) / 1000

print(f"Sparse: {t_sparse*1e6:.0f} us/bracket")
print(f"Dense:  {t_dense*1e6:.0f} us/bracket")
print(f"Speedup: {t_dense/t_sparse:.0f}x")

## PyTorch Differentiable Bracket

The `SparseLieBracket` module wraps the sparse structure constants as PyTorch buffers
and provides a fully differentiable bracket operation with custom backward pass.

In [ ]:
import torch
from equivariant import SparseLieBracket

bracket = SparseLieBracket.from_algebra("E8")
x = torch.randn(248, requires_grad=True)
y = torch.randn(248)

z = bracket(x, y)
loss = z.norm()
loss.backward()

print(f"Input: {x.shape}")
print(f"Output: {z.shape}, norm={z.norm().item():.4f}")
print(f"Gradient: {x.grad.shape}, norm={x.grad.norm().item():.4f}")
print("Autograd works!")

## Equivariance Verification

We verify that the `EquivariantLieConvLayer` commutes with the adjoint action of the Lie group.
For a group element g = exp(ad_omega), equivariance means:

    f(Ad_g x) = Ad_g f(x)

The error should be near zero (limited by float32 precision).

In [ ]:
from equivariant import EquivariantLieConvLayer
from scipy.linalg import expm

layer = EquivariantLieConvLayer(algebra_name="E8")
layer.eval()

# Random adjoint group element
from exceptional import ExceptionalAlgebra
alg = ExceptionalAlgebra("E8")
omega = np.random.randn(248) * 0.01
ad_omega = sum(omega[i] * alg.gen_array[i] for i in range(248))
Ad_g = expm(ad_omega)
Ad_g_t = torch.tensor(Ad_g, dtype=torch.float32)

# Test equivariance
x = torch.randn(5, 248)
edge_index = torch.tensor([[0,1,2,3,4],[1,2,3,4,0]], dtype=torch.long)

with torch.no_grad():
    out_then_transform = layer(x, edge_index) @ Ad_g_t.T
    transform_then_out = layer(x @ Ad_g_t.T, edge_index)

diff = (out_then_transform - transform_then_out).abs().max().item()
print(f"Equivariance error: {diff:.2e}")
print("Equivariant!" if diff < 1e-5 else "NOT equivariant")